# Day 2 - Session 5: Data Manipulation with dplyr
**Duration: ~1 hour**

## Learning Objectives
- Install and load dplyr
- Master the pipe operator (%>%)
- Learn the five main dplyr verbs
- Perform grouped operations
- Chain operations together for efficient data manipulation

## 1. Introduction to dplyr

**dplyr** is part of the tidyverse - a collection of R packages designed for data science. dplyr makes data manipulation:
- **Faster** - optimized for performance
- **More intuitive** - functions named after what they do
- **More readable** - pipe operations flow naturally

### Why dplyr after learning base R?

You've already learned base R methods (subsetting with `[]`, `subset()`, `aggregate()`). dplyr provides a more consistent and readable syntax that becomes especially powerful with complex operations.

In [ ]:
# Install tidyverse (includes dplyr and other useful packages)
# install.packages("tidyverse")  # Uncomment and run once

# Load dplyr
library(dplyr)

# Load sample data
data(starwars)
data(iris)
data(mtcars)

## 2. The Pipe Operator: %>%

The pipe operator (`%>%`) is the key to readable dplyr code. It passes the output of one function as the first argument to the next.

**Think of it as "then":**
- Take the data **then** filter it **then** select columns **then** summarize

**Keyboard shortcut:** Ctrl+Shift+M (Windows) or Cmd+Shift+M (Mac)

In [ ]:
# Without pipe (hard to read)
mean(sqrt(abs(c(-4, -1, 0, 1, 4))))

# With pipe (reads left to right)
c(-4, -1, 0, 1, 4) %>%
  abs() %>%
  sqrt() %>%
  mean()

## 3. The Five Main dplyr Verbs

### 3.1 filter() - Keep Rows That Match Conditions

In [ ]:
# Compare base R vs dplyr

# Base R:
starwars[starwars$species == "Human" & !is.na(starwars$species), ]

# dplyr:
starwars %>%
  filter(species == "Human")

In [ ]:
# Multiple conditions
starwars %>%
  filter(species == "Human" & height > 180)

# OR condition
starwars %>%
  filter(species == "Human" | species == "Droid")

# Using %in%
starwars %>%
  filter(species %in% c("Human", "Droid", "Wookiee"))

### 3.2 select() - Choose Columns to Keep

In [ ]:
# Select specific columns
starwars %>%
  select(name, height, mass)

# Select range
starwars %>%
  select(name:birth_year)

# Exclude columns
starwars %>%
  select(-films, -vehicles, -starships)

In [ ]:
# Helper functions
starwars %>%
  select(starts_with("s"))

starwars %>%
  select(ends_with("color"))

starwars %>%
  select(contains("_"))

### 3.3 mutate() - Create or Modify Columns

In [ ]:
# Add BMI column
starwars %>%
  mutate(bmi = mass / ((height / 100) ^ 2)) %>%
  select(name, height, mass, bmi)

# Multiple new columns
starwars %>%
  mutate(
    height_m = height / 100,
    bmi = mass / (height_m ^ 2),
    bmi_category = ifelse(bmi > 25, "High", "Normal")
  ) %>%
  select(name, bmi, bmi_category)

### 3.4 arrange() - Sort Rows

In [ ]:
# Sort ascending
starwars %>%
  select(name, height) %>%
  arrange(height)

# Sort descending
starwars %>%
  select(name, height) %>%
  arrange(desc(height))

# Sort by multiple columns
starwars %>%
  select(name, species, height) %>%
  arrange(species, desc(height))

### 3.5 summarize() - Calculate Summary Statistics

In [ ]:
# Single summary
starwars %>%
  summarize(mean_height = mean(height, na.rm = TRUE))

# Multiple summaries
starwars %>%
  summarize(
    mean_height = mean(height, na.rm = TRUE),
    sd_height = sd(height, na.rm = TRUE),
    n = n()  # Count rows
  )

## 4. Grouped Operations: The Power of group_by()

**This is where dplyr truly shines!** `group_by()` lets you perform operations separately for each group.

In [ ]:
# Calculate mean height by species
starwars %>%
  group_by(species) %>%
  summarize(
    mean_height = mean(height, na.rm = TRUE),
    n = n()
  ) %>%
  arrange(desc(mean_height))

In [ ]:
# Filter to species with more than 1 character
starwars %>%
  group_by(species) %>%
  summarize(
    count = n(),
    mean_height = mean(height, na.rm = TRUE)
  ) %>%
  filter(count > 1) %>%
  arrange(desc(mean_height))

In [ ]:
# Using mutate with group_by (compare to group average)
starwars %>%
  group_by(species) %>%
  mutate(
    species_mean_height = mean(height, na.rm = TRUE),
    height_diff = height - species_mean_height
  ) %>%
  select(name, species, height, height_diff) %>%
  filter(!is.na(height_diff)) %>%
  head(10)

## 5. count() - Quick Frequency Tables

In [ ]:
# Shortcut for group_by() + summarize(n = n())
starwars %>%
  count(species, sort = TRUE)

# Multiple variables
starwars %>%
  count(species, sex, sort = TRUE)

## 6. Chaining Operations: Where dplyr Shines

The real power is combining multiple operations in a readable pipeline.

In [ ]:
# Complex analysis in one readable chain
starwars %>%
  # Remove rows with missing data
  filter(!is.na(height) & !is.na(mass)) %>%
  # Calculate BMI
  mutate(bmi = mass / ((height / 100) ^ 2)) %>%
  # Keep only certain species
  filter(species %in% c("Human", "Droid")) %>%
  # Select relevant columns
  select(name, species, height, mass, bmi) %>%
  # Sort by BMI
  arrange(desc(bmi))

In [ ]:
# Grouped analysis with multiple steps
iris %>%
  # Calculate areas
  mutate(
    Sepal.Area = Sepal.Length * Sepal.Width,
    Petal.Area = Petal.Length * Petal.Width
  ) %>%
  # Group by species
  group_by(Species) %>%
  # Calculate statistics
  summarize(
    n = n(),
    mean_sepal_area = mean(Sepal.Area),
    mean_petal_area = mean(Petal.Area),
    total_area = mean(Sepal.Area + Petal.Area)
  ) %>%
  # Round values
  mutate(across(where(is.numeric), ~round(., 2))) %>%
  # Sort
  arrange(desc(total_area))

## 7. Comparison: Base R vs dplyr

Let's see the same task in both approaches:

In [ ]:
# Task: For each species in iris, calculate mean Petal.Length 
# for flowers with Sepal.Length > 5, and sort by the result

# BASE R:
iris_subset <- iris[iris$Sepal.Length > 5, ]
result <- aggregate(Petal.Length ~ Species, data = iris_subset, FUN = mean)
result <- result[order(result$Petal.Length, decreasing = TRUE), ]
print(result)

# DPLYR:
iris %>%
  filter(Sepal.Length > 5) %>%
  group_by(Species) %>%
  summarize(mean_petal_length = mean(Petal.Length)) %>%
  arrange(desc(mean_petal_length))

## 8. Practice Exercises

### Exercise 1: Basic dplyr Chain
Using the mtcars dataset:
1. Filter for cars with mpg > 20
2. Select name, mpg, cyl, hp
3. Arrange by hp (descending)
4. Show top 10

In [ ]:
# Your code here
# Hint: Use head(10) at the end or slice_head(n = 10)

### Exercise 2: Grouped Summary
Using the iris dataset:
1. Group by Species
2. Calculate mean and sd of Petal.Length
3. Count observations
4. Arrange by mean Petal.Length

In [ ]:
# Your code here

### Exercise 3: Complex Pipeline
Using the mtcars dataset:
1. Add a column 'efficiency' = mpg / wt
2. Filter to cars with hp > 100
3. Group by cyl
4. Calculate mean efficiency and count
5. Arrange by mean efficiency (descending)

In [ ]:
# Your code here

### Exercise 4: Reproduce This Output

Using starwars dataset, create this exact output:

In [ ]:
# TARGET OUTPUT (run this to see what to reproduce)
starwars %>%
  filter(!is.na(height) & !is.na(mass)) %>%
  filter(height > 150) %>%
  mutate(bmi = mass / ((height/100)^2)) %>%
  select(name, height, mass, bmi, homeworld) %>%
  arrange(desc(bmi)) %>%
  head(5)

In [ ]:
# YOUR CODE: Try to reproduce the output above



## 9. Key Takeaways

### When to use dplyr:
- ✅ Complex data manipulation pipelines
- ✅ Grouped operations
- ✅ When code readability matters
- ✅ Working with data frames

### The dplyr mindset:
1. **Start with your data**
2. **Chain operations** with %>%
3. **Think in verbs**: filter, select, mutate, arrange, summarize
4. **Group when needed** for separate analysis
5. **Read it like a sentence**: Take data, *then* filter, *then* summarize, *then* arrange

### Quick Reference:
- `filter()` - Keep rows
- `select()` - Keep columns  
- `mutate()` - Create/modify columns
- `arrange()` - Sort rows
- `summarize()` - Calculate summaries
- `group_by()` - Group for operations
- `count()` - Quick frequency table
- `%>%` - Pipe operator ("then")

## Summary

In this session, you learned:
- ✅ The pipe operator (%>%) for readable code
- ✅ Five main dplyr verbs: filter, select, mutate, arrange, summarize
- ✅ Grouped operations with group_by()
- ✅ Chaining operations for complex analyses
- ✅ When dplyr is more efficient than base R

**dplyr cheatsheet:** https://github.com/rstudio/cheatsheets/raw/master/data-transformation.pdf

---

## Course Complete! 🎉

You've now learned:
- **Day 1**: R basics, data structures, data manipulation, visualization with ggplot2
- **Day 2**: Best practices, reproducible research, and efficient data manipulation with dplyr

### Continue Your R Journey:
- Practice with your own data
- Explore R packages for your field
- Join the R community online
- Check out "R for Data Science" book (free online)

**Keep coding!** 🚀